> # ⚠️ Do not run this notebook unattended
>
> The last section, **`# Hardware validation`**, connects to the real board and uploads the
> package. *Run All* and `jlx run` reach that section.
>
> - **Safe to run on its own:** everything above `# Hardware validation`. It runs on the host,
>   writes the module and its tests, and needs no board.
> - **Hardware section:** run it one cell at a time, reading the **What to do** line above
>   each cell first.

**Where each cell runs.** Everything above `%board connect` runs in ordinary Python
(CPython, the project `.venv`) on this computer. After `%board connect`, cells run in
MicroPython on the board, except cells that start with `%%host`.

# Hardware and configuration

| | Host run | Hardware run |
|---|---|---|
| **Runs on** | CPython 3.13 in the micropylab kernel, no board | Unexpected Maker FeatherS3 (ESP32-S3), MicroPython 1.28.0, USB `/dev/ttyACM0` |
| **I2C bus** | none | none: the board checks only import the drivers |
| **Result** | not run yet | not run yet |

# TODO:
Nothing open.

Decided (2026-11-16): the registry also maps each driver name to its fake
(`devices/<type>/.../<part>_fake.py`, written by the part's device notebook), so the simulated
bench (`sim.bench`) can build a fake bench from the same routing config as the real one.
Fakes are looked up only on the host; the board never imports them.

# Python file
location: `src/micropython_i2c/devices/__init__.py`

tests: `tests/test_devices.py`

## header

In [ ]:
%%extract ../src/micropython_i2c/devices/__init__.py
"""Concrete devices by type, and the registry mapping config driver names to classes.

    sensors/   environmental/ (TMP102, SHTC3), magnetic/ (TMAG5273), distance/ (VL53L1X, VL53L4CD)
    drivers/   output drivers: QwiicRelay (SparkFun relay)
    muxes/     TCA9548A

Drivers are imported lazily by ``driver_class`` / ``registry()``, so importing one driver on
the board does not pull every other driver into RAM. ``fake_class`` finds a driver's fake for
the simulation (host only).

Generated by ``dev_notebooks/devices_devNB``: edit the notebook, not this file.
"""

# Imports
## External
None: `__import__` is a built-in on both MicroPython and CPython.

## Internal
None at import time: the drivers are imported on demand.

# Setup

Notebook-only: run the test cells in the notebook. `raise_on_error=True` makes a failing test
fail the run.

In [ ]:
import ipytest

ipytest.autoconfig(raise_on_error=True)

# Test file
The test file header: imports only. It is written, not run.

In [ ]:
%%writefile ../tests/test_devices.py
import pytest

from micropython_i2c.devices import DRIVERS, FAKES, driver_class, fake_class, registry
from micropython_i2c.node import I2CDevice, I2CMux

Notebook-only: the imports the validation and test cells use.

In [ ]:
import pytest

from micropython_i2c.node import I2CDevice, I2CMux

# Intent
**Purpose:** a routing table names drivers by string (`"driver": "TMP102"`); this module turns
the string into the class, importing only the drivers a table uses.

**Adding a part:** write its device notebook (driver and fake), then add one line to `DRIVERS`
and one to `FAKES` here, by the module's path under `devices`.

| Name | Driver module | Fake module |
|---|---|---|
| `TMP102` | `sensors.environmental.tmp102` | `sensors.environmental.tmp102_fake` |
| `SHTC3` | `sensors.environmental.shtc3` | `sensors.environmental.shtc3_fake` |
| `TMAG5273` | `sensors.magnetic.tmag5273` | `sensors.magnetic.tmag5273_fake` |
| `VL53L1X` | `sensors.distance.vl53l1x` | `sensors.distance.vl53l1x_fake` |
| `VL53L4CD` | `sensors.distance.vl53l4cd` | `sensors.distance.vl53l4cd_fake` |
| `QwiicRelay` | `drivers.qwiic_relay` | `drivers.qwiic_relay_fake` |
| `TCA9548A` | `muxes.tca9548a` | `muxes.tca9548a_fake` |

## What the module defines

| Name | What it is |
|---|---|
| `DRIVERS` | `{name: (module under devices, class)}` |
| `FAKES` | `{name: (module under devices, fake class)}`, host only |
| `driver_class`, `registry` | one driver class, or `{name: class}` for several |
| `fake_class` | a driver's fake class (host only) |

# Code
## TODO:
Nothing

## Public names

In [ ]:
%%extract ../src/micropython_i2c/devices/__init__.py -a
__all__ = ["DRIVERS", "FAKES", "driver_class", "fake_class", "registry"]

## `DRIVERS` and `FAKES`
Module paths are relative to `micropython_i2c.devices`, so moving a part to another folder is
a one-line change here.

In [ ]:
%%extract ../src/micropython_i2c/devices/__init__.py -a
#: Driver name -> (module under ``micropython_i2c.devices``, class). Add new parts here.
DRIVERS = {
    "TMP102": ("sensors.environmental.tmp102", "TMP102"),
    "SHTC3": ("sensors.environmental.shtc3", "SHTC3"),
    "TMAG5273": ("sensors.magnetic.tmag5273", "TMAG5273"),
    "VL53L1X": ("sensors.distance.vl53l1x", "VL53L1X"),
    "VL53L4CD": ("sensors.distance.vl53l4cd", "VL53L4CD"),
    "QwiicRelay": ("drivers.qwiic_relay", "QwiicRelay"),
    "TCA9548A": ("muxes.tca9548a", "TCA9548A"),
}

#: Driver name -> (module, fake class), for the simulation. Host only.
FAKES = {
    "TMP102": ("sensors.environmental.tmp102_fake", "FakeTMP102"),
    "SHTC3": ("sensors.environmental.shtc3_fake", "FakeSHTC3"),
    "TMAG5273": ("sensors.magnetic.tmag5273_fake", "FakeTMAG5273"),
    "VL53L1X": ("sensors.distance.vl53l1x_fake", "FakeVL53L1X"),
    "VL53L4CD": ("sensors.distance.vl53l4cd_fake", "FakeVL53L4CD"),
    "QwiicRelay": ("drivers.qwiic_relay_fake", "FakeQwiicRelay"),
    "TCA9548A": ("muxes.tca9548a_fake", "FakeTCA9548A"),
}

## `driver_class`, `registry`, `fake_class`
`__import__` with a non-empty `fromlist` returns the leaf module on both MicroPython and
CPython, so one call imports exactly one driver.

In [ ]:
%%extract ../src/micropython_i2c/devices/__init__.py -a
def _load(table, name):
    module, cls = table[name]
    mod = __import__("micropython_i2c.devices." + module, None, None, [cls])
    return getattr(mod, cls)


def driver_class(name):
    """Import and return the driver class registered as ``name``.

    Raises:
        KeyError: If ``name`` is not in ``DRIVERS``.
    """
    return _load(DRIVERS, name)


def registry(names=None):
    """Return ``{name: class}`` for ``names`` (default: every registered driver)."""
    return {n: driver_class(n) for n in (DRIVERS if names is None else names)}


def fake_class(name):
    """Import and return the fake registered for driver ``name`` (host only).

    Raises:
        KeyError: If ``name`` has no fake in ``FAKES``.
    """
    if name not in FAKES:
        raise KeyError("no fake for driver {!r}: write one in its device notebook".format(name))
    return _load(FAKES, name)

### validation/test

#### validation: nominal usage
Every registered driver and fake imports, and each driver is a device or a mux.

In [ ]:
for name in DRIVERS:
    drv, fake = driver_class(name), fake_class(name)
    kind = "mux" if issubclass(drv, I2CMux) else "device"
    print("{:11} {:7} {:40} {}".format(name, kind, drv.__module__, fake.__name__))

#### validation: anti-nominal usage

In [ ]:
try:
    driver_class("BME280")  # not registered
except KeyError as err:
    print("raised KeyError, as expected:\n    {}".format(err))
else:
    raise AssertionError("driver_class('BME280') should have raised KeyError")

In [ ]:
try:
    fake_class("BME280")  # no fake registered
except KeyError as err:
    print("raised KeyError, as expected:\n    {}".format(err))
else:
    raise AssertionError("fake_class('BME280') should have raised KeyError")

#### unit

In [ ]:
%%extract ../tests/test_devices.py -a --strip-ipytest
%%ipytest


def test_registry_imports_every_driver():
    reg = registry()
    assert set(reg) == set(DRIVERS)
    for name, cls in reg.items():
        assert cls.__name__ == DRIVERS[name][1]
        assert issubclass(cls, (I2CDevice, I2CMux))

In [ ]:
%%extract ../tests/test_devices.py -a --strip-ipytest
%%ipytest


def test_registry_of_some_names_and_one_class():
    assert set(registry(["TMP102", "TCA9548A"])) == {"TMP102", "TCA9548A"}
    assert driver_class("SHTC3").__name__ == "SHTC3"

In [ ]:
%%extract ../tests/test_devices.py -a --strip-ipytest
%%ipytest


def test_every_driver_has_a_fake_that_builds_with_no_arguments():
    assert set(FAKES) == set(DRIVERS)
    for name in FAKES:
        cls = fake_class(name)
        assert cls.__name__ == FAKES[name][1]
        cls()  # every fake has defaults for everything

In [ ]:
%%extract ../tests/test_devices.py -a --strip-ipytest
%%ipytest


def test_unknown_names_raise_key_error():
    with pytest.raises(KeyError):
        driver_class("BME280")
    with pytest.raises(KeyError, match="no fake for driver 'BME280'"):
        fake_class("BME280")

# Hardware validation

> ## ⚠️ STOP — do not run this section unattended
>
> **Everything below talks to the real board.** It connects to the FeatherS3 and uploads the
> package. Nothing here uses the I2C bus: it imports every driver on MicroPython.
>
> - Run it **one cell at a time**. Read the **What to do** line above each cell first.
> - The section ends with `%board disconnect`. If you stop partway, run that cell yourself.

## Connect and set up

Notebook-only: connect to the board. Only one notebook can own the board: if another notebook
still has it, run `%board disconnect` there first.

**What to do:** make sure the board is plugged in and no other notebook owns it.

In [ ]:
%board connect

Notebook-only: remove the package an earlier upload left on the board, and make sure `/lib`
exists, so the copy below lands exactly at `/lib/micropython_i2c`.

**What to do:** nothing.

In [ ]:
import os


def rmtree(path):
    """Delete a board folder and everything in it."""
    for entry in os.ilistdir(path):
        child = path + "/" + entry[0]
        if entry[1] == 0x4000:  # folder
            rmtree(child)
        else:
            os.remove(child)
    os.rmdir(path)


try:
    rmtree("/lib/micropython_i2c")
    print("removed the old /lib/micropython_i2c")
except OSError:
    print("no /lib/micropython_i2c on the board yet")
try:
    os.mkdir("/lib")
except OSError:
    pass  # already there

Notebook-only: copy `src/micropython_i2c`, including the file this notebook just wrote, to
`/lib/micropython_i2c`. The copy runs on the host; the path is relative to this notebook's
folder.

**What to do:** nothing. Takes a few seconds.

In [ ]:
%%host
from pathlib import Path

pkg = next(p for p in (Path("../src"), Path("src")) if p.is_dir()) / "micropython_i2c"
%files copy -r {pkg} :/lib/micropython_i2c

Notebook-only: drop any modules an earlier run left on the board, so the new files load, then
import what the checks use and zero the tally.

**What to do:** nothing.

In [ ]:
import sys

for name in [m for m in sys.modules if m.startswith("micropython_i2c")]:
    del sys.modules[name]

from micropython_i2c.debug import check, check_raises, check_reset, check_summary
from micropython_i2c.debug import check_counts  # noqa: F401  (read from the host with %eval)
from micropython_i2c.devices import DRIVERS, driver_class, registry

check_reset()

## Matching tests

**Matches** `test_registry_imports_every_driver`, on MicroPython: each driver module compiles
and imports there (the host cannot show that). Free RAM is printed after each import.

**What to do:** nothing.

In [ ]:
import gc

for name in DRIVERS:
    gc.collect()
    try:
        cls = driver_class(name)
        check("{} imports on MicroPython".format(name), cls.__name__ == DRIVERS[name][1],
              "{} bytes free".format(gc.mem_free()))  # fmt: skip
    except Exception as e:
        check("{} imports on MicroPython".format(name), False, repr(e))

**Matches** `test_unknown_names_raise_key_error` (the driver half: the board never looks up
fakes).

**What to do:** nothing.

In [ ]:
check_raises("test_unknown_names_raise_key_error", KeyError, driver_class, "BME280")
check("test_registry_of_some_names_and_one_class",
      set(registry(["TMP102", "TCA9548A"])) == {"TMP102", "TCA9548A"})  # fmt: skip

## Tally and disconnect

Notebook-only: the PASS/FAIL tally of every check in this section.

**What to do:** read it. Any `FAIL` line above says which check and why.

In [ ]:
check_summary()

Notebook-only: copy the tally to the host before disconnecting, so the last cell can fail the
run after the board is released.

**What to do:** nothing.

In [ ]:
%%host
hw_passed, hw_failed = get_ipython().run_line_magic("eval", "check_counts()")

Notebook-only: release the board, so the next run (or a kernel restart) starts on the host.

**What to do:** nothing.

In [ ]:
%board disconnect

Notebook-only: fail the run if any hardware check failed. It runs on the host.

**What to do:** nothing. An error here means a `FAIL` line above.

In [ ]:
%%host
if hw_failed:
    raise AssertionError("{} of {} hardware checks failed".format(hw_failed, hw_passed + hw_failed))
print("all {} hardware checks passed".format(hw_passed))